# 02 · Generación y carga de datos — Smart Solutions Dietta

**Team Challenge SQL · Parte II** — Alejandro Dietta

Este notebook genera datos sintéticos con **Faker** para las 7 tablas creadas en `01_setup_bigquery.ipynb` y los carga en BigQuery.

| Tabla | Volumen |
|---|---|
| categories | 8 |
| customers | 500 |
| products | 70 |
| orders | 2000 |
| order_items | ~4500 (media de 2–3 productos por pedido) |
| payments | 1 por pedido |
| reviews | ~35 % de las líneas de pedidos entregados |

## 1. Configuración

Misma conexión que en el notebook 01. Se fija una **semilla** para Faker y `random`, de forma que cada ejecución genera exactamente los mismos datos.

In [ ]:
import os
import random
import unicodedata
import warnings
from datetime import date, datetime, time, timedelta
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from faker import Faker
from google.cloud import bigquery
from google.oauth2 import service_account

# google-cloud-bigquery muestra dos avisos informativos que no afectan a la carga ni a las queries:
# - pandas-gbq: en versiones futuras será necesario para cargar DataFrames.
# - BigQuery Storage: sin ese módulo, los resultados se descargan por la API REST normal.
# No se instalan esas librerías porque, al hacerlo, aparecen nuevos avisos de obsolescencia que
# piden sustituir load_table_from_dataframe y client.query().to_dataframe() por las funciones de
# pandas-gbq (to_gbq / read_gbq). Se mantiene el código de la guía del bootcamp y las dependencias
# de requirements.txt, y se silencian solo estos dos avisos concretos.
warnings.filterwarnings("ignore", message=".*pandas-gbq.*")
warnings.filterwarnings("ignore", message=".*BigQuery Storage module not found.*")

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
ROOT = Path(env_path).parent

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = ROOT / os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_PATH)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

Faker.seed(42)
random.seed(42)
fake = Faker("es_ES")

N_CUSTOMERS = 500
N_ORDERS = 2000
REVIEW_RATE = 0.35
START_DATE = date(2024, 1, 1)
END_DATE = date(2025, 12, 31)

## 2. Categorías y productos

El catálogo se define a mano: cada categoría tiene su descripción, un rango de precios realista y sus productos (70 en total).

- `price`: aleatorio dentro del rango de su categoría.
- `cost`: entre el 55 % y el 80 % del precio, así el margen siempre es positivo.
- `stock`: entre 0 y 300 unidades.
- `is_active`: ~90 % de los productos activos.

In [30]:
CATALOG = {
    "Smartphones": {
        "description": "Teléfonos móviles inteligentes",
        "price_range": (199, 1299),
        "products": [
            "Smartphone Nova X5", "Smartphone Nova X5 Pro", "Smartphone Nova X5 Lite",
            "Smartphone Orion S12", "Smartphone Orion S12 Ultra", "Smartphone Pixo 8",
            "Smartphone Pixo 8a", "Smartphone Zenit Z3", "Smartphone Zenit Z3 Mini",
        ],
    },
    "Laptops": {
        "description": "Ordenadores portátiles para trabajo, estudio y gaming",
        "price_range": (499, 2499),
        "products": [
            "Portátil AirBook 13", "Portátil AirBook 15", "Portátil ProBook 14",
            "Portátil ProBook 16", "Portátil WorkMate 14", "Portátil StudyBook 15",
            "Portátil Gamer Blade 15", "Portátil Gamer Blade 17", "Portátil UltraSlim 13",
        ],
    },
    "Tablets": {
        "description": "Tabletas y lectores de libros electrónicos",
        "price_range": (99, 1199),
        "products": [
            "Tablet Nova Tab 11", "Tablet Nova Tab Mini", "Tablet Orion Pad 10",
            "Tablet Orion Pad Pro 12", "Lector eBook Paper 6", "Lector eBook Paper 7",
            "Tablet Kids Tab 8", "Tablet Draw Pad 13",
        ],
    },
    "Audio": {
        "description": "Auriculares, altavoces y barras de sonido",
        "price_range": (29, 399),
        "products": [
            "Auriculares SoundPods", "Auriculares SoundPods Pro", "Auriculares BassMax",
            "Auriculares QuietOne ANC", "Altavoz Boom Mini", "Altavoz Boom XL",
            "Barra de sonido HomeBar", "Barra de sonido HomeBar Plus", "Auriculares RunBuds",
        ],
    },
    "Wearables": {
        "description": "Relojes inteligentes y pulseras de actividad",
        "price_range": (49, 499),
        "products": [
            "Smartwatch Pulse 4", "Smartwatch Pulse 4 Pro", "Smartwatch Active",
            "Pulsera FitBand 6", "Pulsera FitBand 6 Pro", "Smartwatch Kids",
            "Anillo inteligente Ring One", "Smartwatch Outdoor Trek",
        ],
    },
    "Periféricos": {
        "description": "Teclados, ratones, monitores y webcams",
        "price_range": (19, 599),
        "products": [
            "Teclado mecánico KeyPro", "Teclado inalámbrico SlimKeys", "Ratón ErgoMouse",
            "Ratón inalámbrico ClickAir", "Monitor 24 ViewFull", "Monitor 27 ViewFull QHD",
            "Monitor 32 ViewPro 4K", "Webcam StreamCam HD", "Micrófono USB VoiceCast",
        ],
    },
    "Gaming": {
        "description": "Consolas, mandos y accesorios de juego",
        "price_range": (29, 549),
        "products": [
            "Consola PlayBox S", "Consola PlayBox X", "Consola portátil GoPlay",
            "Mando inalámbrico GamePad", "Auriculares gaming HeadShot", "Silla gaming ThroneX",
            "Volante RaceWheel", "Teclado gaming Striker RGB", "Ratón gaming Viper",
        ],
    },
    "Accesorios": {
        "description": "Cargadores, cables, fundas y almacenamiento",
        "price_range": (9, 129),
        "products": [
            "Cargador USB-C 65W", "Cargador inalámbrico MagPad", "Cable USB-C 2m",
            "Funda protectora smartphone", "Protector de pantalla de cristal", "Power bank 20000mAh",
            "Tarjeta microSD 256GB", "SSD externo 1TB", "Hub USB-C 7 en 1",
        ],
    },
}

categories_rows = []
products_rows = []
category_id = 0
product_id = 0

for category_name, info in CATALOG.items():
    # Una fila en categories por cada categoría del catálogo
    category_id += 1
    categories_rows.append({
        "category_id": category_id,
        "name": category_name,
        "description": info["description"],
    })

    # Una fila en products por cada producto de esa categoría
    min_price = info["price_range"][0]
    max_price = info["price_range"][1]
    for product_name in info["products"]:
        product_id += 1
        price = round(random.uniform(min_price, max_price), 2)
        cost = round(price * random.uniform(0.55, 0.80), 2)
        products_rows.append({
            "product_id": product_id,
            "category_id": category_id,
            "name": product_name,
            "price": price,
            "cost": cost,
            "stock": random.randint(0, 300),
            "is_active": random.random() < 0.9,
        })

categories = pd.DataFrame(categories_rows)
products = pd.DataFrame(products_rows)

print(f"{len(categories)} categorías, {len(products)} productos")
products.head()

8 categorías, 70 productos


,product_id,category_id,name,price,cost,stock,is_active
0,1,1,Smartphone Nova X5,902.37,501.95,140,True
1,2,1,Smartphone Nova X5 Pro,352.49,202.90,279,True
2,3,1,Smartphone Nova X5 Lite,663.11,369.65,111,True
3,4,1,Smartphone Orion S12,861.22,594.51,279,True
4,5,1,Smartphone Orion S12 Ultra,693.13,429.43,3,True


## 3. Clientes

Cada cliente se genera con el *locale* de Faker de su país, para que nombre y teléfono sean coherentes (un cliente de Francia tiene nombre y teléfono franceses).

- `city`: una de las 6 ciudades principales de su país. Así varios clientes comparten ciudad y el análisis geográfico tiene sentido.

- `email`: nombre + apellido + id, sin tildes ni espacios → siempre único.
- `phone`: opcional, ~10 % de clientes sin teléfono.
- `acquisition_channel`: organic, paid_ads, social_media, referral o email.
- `registration_date`: entre 01/01/2024 y 30/06/2025.

In [31]:
COUNTRIES = {
    "España": "es_ES",
    "Francia": "fr_FR",
    "Alemania": "de_DE",
    "Italia": "it_IT",
    "Portugal": "pt_PT",
}
CITIES = {
    "España": ["Madrid", "Barcelona", "Valencia", "Sevilla", "Bilbao", "Zaragoza"],
    "Francia": ["París", "Lyon", "Marsella", "Toulouse", "Burdeos", "Lille"],
    "Alemania": ["Berlín", "Múnich", "Hamburgo", "Colonia", "Fráncfort", "Stuttgart"],
    "Italia": ["Roma", "Milán", "Nápoles", "Turín", "Florencia", "Bolonia"],
    "Portugal": ["Lisboa", "Oporto", "Braga", "Coímbra", "Faro", "Aveiro"],
}
COUNTRY_WEIGHTS = [40, 20, 20, 10, 10]
CHANNELS = ["organic", "paid_ads", "social_media", "referral", "email"]
CHANNEL_WEIGHTS = [35, 25, 20, 10, 10]

fakers = {country: Faker(locale) for country, locale in COUNTRIES.items()}


def to_ascii(text):
    """Quita tildes y espacios para construir el email."""
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()
    return text.lower().replace(" ", "")


customers_rows = []
for customer_id in range(1, N_CUSTOMERS + 1):
    country = random.choices(list(COUNTRIES), weights=COUNTRY_WEIGHTS)[0]
    local_fake = fakers[country]
    first_name = local_fake.first_name()
    last_name = local_fake.last_name()
    customers_rows.append({
        "customer_id": customer_id,
        "first_name": first_name,
        "last_name": last_name,
        "email": f"{to_ascii(first_name)}.{to_ascii(last_name)}{customer_id}@{local_fake.free_email_domain()}",
        "phone": local_fake.phone_number() if random.random() < 0.9 else None,
        "country": country,
        "city": random.choice(CITIES[country]),
        "acquisition_channel": random.choices(CHANNELS, weights=CHANNEL_WEIGHTS)[0],
        "registration_date": fake.date_between(START_DATE, date(2025, 6, 30)),
    })

customers = pd.DataFrame(customers_rows)
customers.head()

,customer_id,first_name,last_name,email,phone,country,city,acquisition_channel,registration_date
0,1,Lucie,Marie,lucie.marie1@voila.fr,0382321819,Francia,Lille,paid_ads,2024-11-23
1,2,Anton,Becker,anton.becker2@aol.de,+49(0) 389083863,Alemania,Stuttgart,organic,2024-12-14
2,3,Bianca,Lettiere,bianca.lettiere3@libero.it,+39 363654235,Italia,Roma,social_media,2024-10-22
3,4,Cleto,Palomar,cleto.palomar4@yahoo.com,+34 884 078 161,España,Barcelona,social_media,2025-03-04
4,5,Lázaro,Pardo,lazaro.pardo5@hotmail.com,+34871103413,España,Valencia,paid_ads,2024-03-19


## 4. Pedidos

- `order_date`: siempre **posterior** al registro del cliente y como máximo el 31/12/2025.
- `status`: la mayoría `delivered`; pocos `cancelled` o `returned`.
- Fechas según el estado, siempre en orden **pedido → envío → entrega**:

| status | shipped_date | delivered_date |
|---|---|---|
| pending, confirmed, cancelled | — | — |
| shipped | 1–3 días tras el pedido | — |
| delivered, returned | 1–3 días tras el pedido | 1–7 días tras el envío |

- Envío a la ciudad y país del cliente, con una dirección generada por Faker.

In [32]:
STATUSES = ["pending", "confirmed", "shipped", "delivered", "cancelled", "returned"]
STATUS_WEIGHTS = [5, 5, 8, 70, 7, 5]

orders_rows = []
for order_id in range(1, N_ORDERS + 1):
    customer = customers_rows[random.randrange(N_CUSTOMERS)]
    order_date = fake.date_time_between(
        start_date=datetime.combine(customer["registration_date"], time()),
        end_date=datetime.combine(END_DATE, time(23, 59)),
    )
    status = random.choices(STATUSES, weights=STATUS_WEIGHTS)[0]

    shipped_date = None
    delivered_date = None
    if status in ("shipped", "delivered", "returned"):
        shipped_date = order_date.date() + timedelta(days=random.randint(1, 3))
    if status in ("delivered", "returned"):
        delivered_date = shipped_date + timedelta(days=random.randint(1, 7))

    orders_rows.append({
        "order_id": order_id,
        "customer_id": customer["customer_id"],
        "order_date": order_date,
        "status": status,
        "shipping_address": fakers[customer["country"]].street_address(),
        "shipping_city": customer["city"],
        "shipping_country": customer["country"],
        "shipped_date": shipped_date,
        "delivered_date": delivered_date,
    })

orders = pd.DataFrame(orders_rows)
print(orders["status"].value_counts())
orders.head()

status
delivered    1382
shipped       188
cancelled     135
pending       106
confirmed     100
returned       89
Name: count, dtype: int64


,order_id,customer_id,order_date,status,shipping_address,shipping_city,shipping_country,shipped_date,delivered_date
0,1,269,2025-10-23 11:04:08,pending,"157, boulevard de Gillet",Lille,Francia,None,None
1,2,434,2025-04-04 02:36:35,confirmed,Wulfstr. 7518,Colonia,Alemania,None,None
2,3,419,2024-09-14 11:25:23,delivered,Elfie-Holzapfel-Ring 6-2,Múnich,Alemania,2024-09-15,2024-09-19
3,4,71,2025-04-15 09:10:25,delivered,"R. Castro, S/N",Lisboa,Portugal,2025-04-16,2025-04-17
4,5,363,2024-11-03 23:01:34,shipped,Glorieta de Conrado Antón 967,Barcelona,España,2024-11-04,None


## 5. Líneas de pedido

- Cada pedido tiene entre 1 y 4 productos **distintos** (media ≈ 2,3 → ~4500 líneas).
- `unit_price`: precio del producto ±5 %, porque el precio en el momento de la compra puede diferir del precio actual.
- `discount_pct`: la mayoría sin descuento; algunos con 5 %, 10 % o 15 %.

In [33]:
ITEMS_PER_ORDER = [1, 2, 3, 4]
ITEMS_WEIGHTS = [25, 35, 25, 15]
DISCOUNTS = [0.0, 0.05, 0.10, 0.15]
DISCOUNT_WEIGHTS = [70, 15, 10, 5]

order_items_rows = []
for order in orders_rows:
    n_items = random.choices(ITEMS_PER_ORDER, weights=ITEMS_WEIGHTS)[0]
    for product in random.sample(products_rows, n_items):
        order_items_rows.append({
            "order_item_id": len(order_items_rows) + 1,
            "order_id": order["order_id"],
            "product_id": product["product_id"],
            "quantity": random.choices([1, 2, 3], weights=[80, 15, 5])[0],
            "unit_price": round(product["price"] * random.uniform(0.95, 1.05), 2),
            "discount_pct": random.choices(DISCOUNTS, weights=DISCOUNT_WEIGHTS)[0],
        })

order_items = pd.DataFrame(order_items_rows)
print(f"{len(order_items)} líneas, media de {len(order_items) / len(orders):.2f} productos por pedido")
order_items.head()

4538 líneas, media de 2.27 productos por pedido


,order_item_id,order_id,product_id,quantity,unit_price,discount_pct
0,1,1,45,1,91.38,0.00
1,2,2,58,1,281.37,0.05
2,3,2,44,1,61.67,0.00
3,4,3,58,2,277.14,0.05
4,5,3,20,1,642.29,0.00


## 6. Pagos

Un pago por pedido.

- `amount`: suma de las líneas del pedido con el descuento aplicado → `quantity × unit_price × (1 − discount_pct)`.
- `status` coherente con el estado del pedido:

| Pedido | Pago |
|---|---|
| confirmed, shipped, delivered | completed |
| pending | pending |
| cancelled | failed o refunded |
| returned | refunded |

- `payment_date`: entre 1 y 30 minutos después del pedido.

In [34]:
PAYMENT_METHODS = ["credit_card", "paypal", "bank_transfer", "bizum"]
PAYMENT_METHOD_WEIGHTS = [50, 25, 10, 15]

line_totals = order_items["quantity"] * order_items["unit_price"] * (1 - order_items["discount_pct"])
order_totals = line_totals.groupby(order_items["order_id"]).sum().round(2)


def payment_status(order_status):
    if order_status == "pending":
        return "pending"
    if order_status == "cancelled":
        return random.choice(["failed", "refunded"])
    if order_status == "returned":
        return "refunded"
    return "completed"


payments_rows = []
for order in orders_rows:
    payments_rows.append({
        "payment_id": order["order_id"],
        "order_id": order["order_id"],
        "payment_method": random.choices(PAYMENT_METHODS, weights=PAYMENT_METHOD_WEIGHTS)[0],
        "status": payment_status(order["status"]),
        "amount": order_totals[order["order_id"]],
        "payment_date": order["order_date"] + timedelta(minutes=random.randint(1, 30)),
    })

payments = pd.DataFrame(payments_rows)
print(payments["status"].value_counts())
payments.head()

status
completed    1670
refunded      151
pending       106
failed         73
Name: count, dtype: int64


,payment_id,order_id,payment_method,status,amount,payment_date
0,1,1,bank_transfer,pending,91.38,2025-10-23 11:21:08
1,2,2,credit_card,completed,328.97,2025-04-04 02:57:35
2,3,3,paypal,completed,1168.86,2024-09-14 11:32:23
3,4,4,paypal,completed,708.02,2025-04-15 09:15:25
4,5,5,credit_card,completed,1192.00,2024-11-03 23:03:34


## 7. Valoraciones

Se valoran ~35 % de las líneas de pedido **entregadas** (`status = 'delivered'`), una valoración por línea.

- `rating`: de 1 a 5, con más peso en 4 y 5.
- `comment`: opcional (~60 % de las valoraciones), elegido de una lista de frases según la nota.
- `review_date`: entre 1 y 30 días después de la entrega.

In [35]:
RATING_WEIGHTS = [5, 8, 15, 32, 40]
COMMENTS = {
    1: ["No funciona como esperaba.", "Mala calidad, lo he devuelto.", "Muy decepcionado con la compra."],
    2: ["Calidad regular para el precio.", "Llegó bien pero no cumple lo que promete."],
    3: ["Cumple, sin más.", "Correcto por el precio.", "Está bien, aunque esperaba algo mejor."],
    4: ["Muy buen producto, lo recomiendo.", "Buena calidad y envío rápido.", "Contento con la compra."],
    5: ["¡Excelente! Superó mis expectativas.", "Perfecto, repetiría sin dudarlo.", "Calidad top y llegó antes de lo previsto."],
}

delivered_dates = {order["order_id"]: order["delivered_date"] for order in orders_rows if order["status"] == "delivered"}
delivered_items = [item for item in order_items_rows if item["order_id"] in delivered_dates]
reviewed_items = random.sample(delivered_items, round(len(delivered_items) * REVIEW_RATE))

reviews_rows = []
for item in reviewed_items:
    rating = random.choices([1, 2, 3, 4, 5], weights=RATING_WEIGHTS)[0]
    reviews_rows.append({
        "review_id": len(reviews_rows) + 1,
        "order_item_id": item["order_item_id"],
        "rating": rating,
        "comment": random.choice(COMMENTS[rating]) if random.random() < 0.6 else None,
        "review_date": delivered_dates[item["order_id"]] + timedelta(days=random.randint(1, 30)),
    })

reviews = pd.DataFrame(reviews_rows)
print(f"{len(reviews)} valoraciones de {len(delivered_items)} líneas entregadas ({len(reviews) / len(delivered_items):.0%})")
reviews.head()

1115 valoraciones de 3187 líneas entregadas (35%)


,review_id,order_item_id,rating,comment,review_date
0,1,2372,4,Contento con la compra.,2026-01-13
1,2,90,1,No funciona como esperaba.,2025-09-01
2,3,1910,5,¡Excelente! Superó mis expectativas.,2025-11-17
3,4,3564,4,"Muy buen producto, lo recomiendo.",2025-12-07
4,5,3636,5,NaN,2025-11-12


## 8. Carga en BigQuery

`load_table` sigue el patrón de la guía (`load_table_from_dataframe` con `WRITE_TRUNCATE`, que sobrescribe la tabla y permite repetir la carga sin duplicar datos):

- El esquema se lee de la tabla creada en el notebook 01.
- Si la carga falla, se captura el error y se muestra.
- **Validación:** tras la carga se compara el `COUNT(*)` de BigQuery con las filas del DataFrame.

In [36]:
def load_table(df, table_name):
    table_ref = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    try:
        schema = client.get_table(table_ref).schema
        job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE", schema=schema)
        job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
        job.result()  # Esperar a que termine
    except Exception as error:
        print(f"❌ Error al cargar {table_name}: {error}")
        return False

    loaded_rows = client.query(f"SELECT COUNT(*) AS total FROM `{table_ref}`").to_dataframe()["total"][0]
    if loaded_rows != len(df):
        print(f"❌ {table_name}: {len(df)} filas generadas pero {loaded_rows} en BigQuery")
        return False

    print(f"✅ {table_name}: {loaded_rows} filas cargadas y validadas")
    return True

Se cargan las tablas en el mismo orden de dependencias con el que se crearon.

In [37]:
TABLES = {
    "categories": categories,
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
}

for table_name, df in TABLES.items():
    load_table(df, table_name)

c:\Users\Usuario\Documents\Code\TheBridge\tc-sql-adietta\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(
c:\Users\Usuario\Documents\Code\TheBridge\tc-sql-adietta\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ categories: 8 filas cargadas y validadas


c:\Users\Usuario\Documents\Code\TheBridge\tc-sql-adietta\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(
c:\Users\Usuario\Documents\Code\TheBridge\tc-sql-adietta\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ customers: 500 filas cargadas y validadas


c:\Users\Usuario\Documents\Code\TheBridge\tc-sql-adietta\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(
c:\Users\Usuario\Documents\Code\TheBridge\tc-sql-adietta\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


...

c:\Users\Usuario\Documents\Code\TheBridge\tc-sql-adietta\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(
c:\Users\Usuario\Documents\Code\TheBridge\tc-sql-adietta\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ payments: 2000 filas cargadas y validadas


c:\Users\Usuario\Documents\Code\TheBridge\tc-sql-adietta\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ reviews: 1115 filas cargadas y validadas


c:\Users\Usuario\Documents\Code\TheBridge\tc-sql-adietta\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
